In [ ]:
import os
import requests
import pandas as pd

from netsuite_auth import get_auth, run_suiteql

auth, BASE_URL = get_auth()

# Safety check — default behavior requires sandbox to prevent accidental production runs.
# Remove or adjust this check if you intentionally need to delete from production.
_env = os.environ.get("NETSUITE_ENV", "sandbox2")
if "sandbox" not in _env.lower():
    raise RuntimeError(
        f"NETSUITE_ENV is '{_env}' — this notebook deletes records. "
        "Set NETSUITE_ENV to a sandbox environment, or remove this check if you intend to run against production."
    )

# Derive the REST Record API base URL from the SuiteQL URL
RECORD_API_URL = BASE_URL.replace("/services/rest/query/v1/suiteql", "/services/rest/record/v1")

In [ ]:
# --- Find transactions to delete ---
# Adjust TRANSACTION_TYPES and the date range to match what you want to remove.
#
# Common type codes:
#   CustInvc  = Invoice
#   CustCred  = Credit Memo
#   SalesOrd  = Sales Order
#   PurchOrd  = Purchase Order
#   VendBill  = Vendor Bill
#   ItemShip  = Item Fulfillment
#   ItemRcpt  = Item Receipt
#   Journal   = Journal Entry

TRANSACTION_TYPES = ["CustInvc", "CustCred"]  # types to delete
CREATED_AFTER     = "2025-01-01"              # trandate >= this date (YYYY-MM-DD)
CREATED_BEFORE    = "2025-01-31"              # trandate <= this date (YYYY-MM-DD)

type_list = ", ".join(f"'{t}'" for t in TRANSACTION_TYPES)

results = run_suiteql(f"""
    SELECT t.id, t.type, t.tranid, t.trandate
    FROM transaction t
    WHERE t.trandate >= TO_DATE('{CREATED_AFTER}',  'YYYY-MM-DD')
      AND t.trandate <= TO_DATE('{CREATED_BEFORE}', 'YYYY-MM-DD')
      AND t.type IN ({type_list})
      AND t.voided = 'F'
    ORDER BY t.type, t.id
""", limit=1000)

df = pd.DataFrame(results)
print(f"Found {len(df)} transaction(s) matching the filter")
print("Review this list carefully before running the delete cell below.")
df

In [ ]:
# --- Delete transactions ---
# Set DRY_RUN = True to preview what would be deleted without actually deleting anything.
# When you're ready, set DRY_RUN = False.
#
# If deleting both credit memos and invoices: delete credit memos first.
# NetSuite blocks deleting an invoice that has a linked, non-voided credit memo.

DRY_RUN = True

# Map NetSuite type codes to REST Record API path segments
TYPE_PATH = {
    "CustInvc": "invoice",
    "CustCred": "creditmemo",
    "SalesOrd": "salesorder",
    "PurchOrd": "purchaseorder",
    "VendBill": "vendorbill",
    "ItemShip": "itemfulfillment",
    "ItemRcpt": "itemreceipt",
    "Journal":  "journalentry",
}

def delete_transaction(txn_id: str, txn_type: str, tranid: str) -> bool:
    path     = TYPE_PATH.get(txn_type, txn_type.lower())
    url      = f"{RECORD_API_URL}/{path}/{txn_id}"
    response = requests.delete(url, auth=auth)
    if response.ok:
        print(f"  Deleted {path} {tranid} (ID {txn_id})")
        return True
    else:
        print(f"  FAILED {path} {tranid} (ID {txn_id}): {response.status_code} — {response.text}")
        return False


if df.empty:
    print("Nothing to delete.")
else:
    # Sort: credit memos before invoices so linked CMs are deleted first
    TYPE_ORDER = {"CustCred": 0, "CustInvc": 1}
    df_sorted = df.sort_values(by="type", key=lambda s: s.map(lambda x: TYPE_ORDER.get(x, 99)))

    if DRY_RUN:
        print("DRY RUN — no records will be deleted")
        for _, row in df_sorted.iterrows():
            path = TYPE_PATH.get(row["type"], row["type"].lower())
            print(f"  Would delete {path} {row['tranid']} (ID {row['id']})")
    else:
        errors = 0
        for _, row in df_sorted.iterrows():
            if not delete_transaction(row["id"], row["type"], row["tranid"]):
                errors += 1

        print(f"\nDone — {len(df) - errors}/{len(df)} deleted, {errors} error(s)")